# 04. MLOps Drift Monitoring & Automated Model Retraining (Phase 8)

This notebook implements and demonstrates production drift monitoring for the GitHub Issue Triage predictive agent:
1. **Statistical Drift Measurement**: Population Stability Index (PSI) and two-sample Kolmogorov-Smirnov (KS) tests across text, diagnostic, and form features.
2. **Threshold Calibration**:
   - `PSI < 0.10`: Stable distribution (Normal / Green)
   - `0.10 <= PSI < 0.25`: Moderate shift / Warning (Yellow)
   - `PSI >= 0.25`: Severe drift / Retraining trigger (Red)
3. **Simulated Production Distribution Shift**: Simulating an influx of terse, vague post-release bug reports lacking tracebacks.
4. **Visualizing Distribution Shifts**: Comparative probability density and empirical histograms.
5. **Automated Retraining Decision Engine**: Integrating drift alerts, cumulative volume thresholds, and empirical Brier score degradation rules.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is on sys.path
candidate_root = os.path.abspath("..")
if candidate_root not in sys.path:
    sys.path.insert(0, candidate_root)

from triage.monitoring.drift import calculate_psi, calculate_ks_test, run_drift_analysis
from triage.monitoring.retrain import evaluate_retraining_triggers, PSI_RETRAIN_THRESHOLD

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (12, 5)
print("Libraries imported successfully.")

## 1. Reference vs Recent Inference Cohort Data Loading

We load the verified Home Assistant Core feature dataset (`data/interim/features.parquet`) and perform a chronological split:
- **Reference Cohort (80%)**: Baseline training distribution.
- **Recent Cohort (20%)**: Operational inference monitoring window.

In [ ]:
features_path = os.path.join("..", "data", "interim", "features.parquet")
if not os.path.exists(features_path):
    features_path = os.path.join("data", "interim", "features.parquet")

df = pd.read_parquet(features_path)
df["created_at_dt"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
df = df.dropna(subset=["created_at_dt"]).sort_values("created_at_dt")

split_idx = int(len(df) * 0.80)
ref_df = df.iloc[:split_idx].copy()
curr_df = df.iloc[split_idx:].copy()

print(f"Total Tracked Records: {len(df)}")
print(f"  Reference Baseline Cohort: {len(ref_df)} tickets ({ref_df['created_at_dt'].min().date()} to {ref_df['created_at_dt'].max().date()})")
print(f"  Recent Monitoring Cohort:  {len(curr_df)} tickets ({curr_df['created_at_dt'].min().date()} to {curr_df['created_at_dt'].max().date()})")

## 2. Empirical Baseline Drift Metrics (Production Features)

We evaluate PSI and Kolmogorov-Smirnov statistics across key triage features:
- `body_len` & `title_len`: Descriptive detail and brevity.
- `has_traceback` & `has_logs`: Diagnostic completeness flags.
- `n_code_blocks` & `n_urls`: Technical reproduction signals.
- `n_missing_form_fields`: Adherence to template guidelines.

In [ ]:
features_to_monitor = [
    "title_len",
    "body_len",
    "n_code_blocks",
    "n_urls",
    "has_traceback",
    "has_logs",
    "n_missing_form_fields"
]

results = []
for feat in features_to_monitor:
    if feat in ref_df.columns and feat in curr_df.columns:
        exp_vals = ref_df[feat].dropna().astype(float).values
        act_vals = curr_df[feat].dropna().astype(float).values
        
        psi = calculate_psi(exp_vals, act_vals)
        ks_stat, p_val = calculate_ks_test(exp_vals, act_vals)
        
        status = "Normal" if psi < 0.10 else ("Warning" if psi < 0.25 else "Retrain Alert")
        results.append({
            "Feature": feat,
            "PSI Score": psi,
            "KS Statistic": ks_stat,
            "KS p-value": p_val,
            "Ref Mean": np.round(np.mean(exp_vals), 2),
            "Curr Mean": np.round(np.mean(act_vals), 2),
            "Status": status
        })

drift_summary_table = pd.DataFrame(results)
print(drift_summary_table.to_string(index=False))

## 3. [SIMULATION] Simulated Distribution Shift (Post-Release Terse Bug Influx)

> [!IMPORTANT]
> **SIMULATION EXPERIMENT (SYNTHETIC BENCHMARK - NOT REAL DATA)**: The distribution shift evaluated in this section is an operational simulation created by synthetically perturbing recent issue records. This simulates an influx of terse, post-release bug reports to validate that our monitoring pipeline triggers appropriate drift warnings and automated retraining alerts.

In production software ecosystems, major version releases frequently induce a severe distribution shift:
- Hundreds of end-users submit terse one-liner issues (< 15 words) stating *"Integration stopped working after update"*.
- Absence of logs and tracebacks spikes from ~16% to > 70%.
- Missing form fields rise dramatically.

We simulate this real-world operational shift to test our monitoring detection sensitivity.


In [ ]:
# Generate synthetic shifted cohort reflecting a major release surge of terse reports
shifted_df = curr_df.copy()
np.random.seed(42)

# 60% of issues become terse one-liners without tracebacks or code blocks
shift_mask = np.random.rand(len(shifted_df)) < 0.60
shifted_df.loc[shift_mask, "body_len"] = np.random.exponential(scale=15, size=shift_mask.sum()) + 5
shifted_df.loc[shift_mask, "has_traceback"] = 0
shifted_df.loc[shift_mask, "has_logs"] = 0
shifted_df.loc[shift_mask, "n_code_blocks"] = 0
shifted_df.loc[shift_mask, "n_missing_form_fields"] = np.random.choice([2, 3, 4], size=shift_mask.sum())

print(f"[SIMULATION] Injected synthetic post-release shift on {shift_mask.sum()}/{len(shifted_df)} recent tickets (Simulation Benchmark - Not Real Data).")


In [ ]:
shifted_results = []
for feat in features_to_monitor:
    exp_vals = ref_df[feat].dropna().astype(float).values
    act_vals = shifted_df[feat].dropna().astype(float).values
    
    psi = calculate_psi(exp_vals, act_vals)
    ks_stat, p_val = calculate_ks_test(exp_vals, act_vals)
    
    status = "Normal" if psi < 0.10 else ("Warning" if psi < 0.25 else "Retrain Alert")
    shifted_results.append({
        "Feature": feat,
        "Shifted PSI": psi,
        "KS Statistic": ks_stat,
        "KS p-value": p_val,
        "Ref Mean": np.round(np.mean(exp_vals), 2),
        "Shifted Mean": np.round(np.mean(act_vals), 2),
        "Alert Triggered": status
    })

shifted_summary_table = pd.DataFrame(shifted_results)
print(shifted_summary_table.to_string(index=False))

## 4. Visualizing Distribution Shifts (Histograms & Frequency Densities)

We plot the comparative distributions before vs after the simulated shift for:
1. `body_len` (Text Brevity shift)
2. `n_missing_form_fields` (Form Completeness shift)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Plot 1: body_len distribution
b_psi = shifted_summary_table.loc[shifted_summary_table['Feature']=='body_len', 'Shifted PSI'].values[0]
axes[0].hist(ref_df['body_len'].clip(upper=200), bins=30, density=True, alpha=0.5, color='#2563eb', label='Baseline Reference')
axes[0].hist(shifted_df['body_len'].clip(upper=200), bins=30, density=True, alpha=0.5, color='#dc2626', label='Shifted Post-Release (Simulated)')
axes[0].set_title(f'Body Word Length Drift (PSI = {b_psi:.4f})', fontweight='bold')
axes[0].set_xlabel('Body Length (Words)')
axes[0].set_ylabel('Probability Density')
axes[0].legend()

# Plot 2: n_missing_form_fields distribution
m_psi = shifted_summary_table.loc[shifted_summary_table['Feature']=='n_missing_form_fields', 'Shifted PSI'].values[0]
bins = np.arange(-0.5, 6.5, 1)
axes[1].hist(ref_df['n_missing_form_fields'], bins=bins, density=True, alpha=0.5, color='#2563eb', label='Baseline Reference')
axes[1].hist(shifted_df['n_missing_form_fields'], bins=bins, density=True, alpha=0.5, color='#dc2626', label='Shifted Post-Release (Simulated)')
axes[1].set_title(f'Missing Form Fields Drift (PSI = {m_psi:.4f})', fontweight='bold')
axes[1].set_xlabel('Missing Form Fields Count')
axes[1].set_ylabel('Relative Frequency')
axes[1].legend()

plt.tight_layout()
os.makedirs('images', exist_ok=True)
plot_path = os.path.join('images', 'drift_distribution_comparison.png')
plt.savefig(plot_path, dpi=120)
plt.show()
print(f'Comparative drift visualization saved to {plot_path}')

## 5. Automated Retraining Decision Engine Execution

We execute `evaluate_retraining_triggers()` to demonstrate automated decision-making:
- Evaluates statistical feature drift against threshold `PSI >= 0.25`.
- Checks empirical Brier score error degradation.
- Checks cumulative resolved issue volume threshold (>= 500 records).
- Generates structured decision artifact: `data/monitoring/retrain_decision.json`.

In [ ]:
# Evaluate automated retraining triggers on the shifted scenario
shifted_drift_summary = {
    "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    "status": "completed",
    "reference_records": len(ref_df),
    "current_records": len(shifted_df),
    "features_analyzed": {r["Feature"]: {"psi": r["Shifted PSI"], "alert_level": "alert" if r["Shifted PSI"] >= 0.25 else "normal"} for r in shifted_results},
    "overall_drift_alert": True
}

retrain_decision = evaluate_retraining_triggers(
    drift_summary=shifted_drift_summary,
    features_path=features_path
)

print(json.dumps(retrain_decision, indent=2))